[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014-integer-programming/blob/main/notebooks/week1_knapsack.ipynb)

In [ ]:
%pip install -q gurobipy

# Knapsack, solved via Gurobi WLS

A normal Gurobi license is a file (`gurobi.lic`) installed on one machine. **WLS** instead lets you authenticate with three plain text values, so it works anywhere — a classroom laptop with nothing installed, a fresh Colab runtime, a Docker container — with no file to install at all.

A WLS credential has three fields:

| field | looks like |
|---|---|
| `WLSACCESSID` | `203dec48-e3f8-46ac-0184-92d7d6ded944` |
| `WLSSECRET`   | `a080cce8-4e01-4e36-955e-61592c5630db` |
| `LICENSEID`   | `12127` (a number) |

**Get your own for free:** log in to the [Gurobi user portal](https://portal.gurobi.com) with your academic email, request a free academic WLS license, and copy the three values it gives you. **Each student should have their own** — don't reuse one classmate's credentials, and never paste real credentials into a notebook you save or share.

## Setup: read your WLS credentials

The cell below auto-detects Colab. On Colab it pulls your three values from **Secrets** (key icon in the left sidebar); locally it reads them from environment variables you set yourself. Either way, this notebook file never has a real secret written into it, so it stays safe to save and share.

In [ ]:
import os

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Add these three as Colab Secrets first (key icon in the left sidebar),
    # then switch on "Notebook access" for each -- values never get saved
    # into this notebook's file.
    os.environ["WLSACCESSID"] = userdata.get("WLSACCESSID")
    os.environ["WLSSECRET"] = userdata.get("WLSSECRET")
    os.environ["LICENSEID"] = userdata.get("LICENSEID")
else:
    # Running locally: set these once for this session (uncomment) --
    # os.environ["WLSACCESSID"] = "paste-your-own-access-id"
    # os.environ["WLSSECRET"]   = "paste-your-own-secret"
    # os.environ["LICENSEID"]   = "paste-your-own-license-id"
    pass

missing = [k for k in ("WLSACCESSID", "WLSSECRET", "LICENSEID") if k not in os.environ]
if missing:
    raise RuntimeError(f"Set these environment variables first: {missing}")
print("WLS credentials found in the environment.")

In [ ]:
import gurobipy as gp
from gurobipy import GRB

options = {
    "WLSACCESSID": os.environ["WLSACCESSID"],
    "WLSSECRET": os.environ["WLSSECRET"],
    "LICENSEID": int(os.environ["LICENSEID"]),
}
env = gp.Env(params=options)   # kept open (no `with`) so later cells can reuse it

## The knapsack model — explicit form

Same instance as `ip-games/knapsack.html`: 6 items, one binary decision variable each, one weight constraint. Everything is typed out by hand — no dictionaries, no loops, no `gp.quicksum` — so you can see every term of the objective and the constraint exactly as it appears on paper.

In [ ]:
m = gp.Model("knapsack", env=env)

# decision variables -- one binary per item
x_rifle   = m.addVar(vtype=GRB.BINARY, name="x_rifle")
x_boots   = m.addVar(vtype=GRB.BINARY, name="x_boots")
x_helmet  = m.addVar(vtype=GRB.BINARY, name="x_helmet")
x_canteen = m.addVar(vtype=GRB.BINARY, name="x_canteen")
x_medkit  = m.addVar(vtype=GRB.BINARY, name="x_medkit")
x_rations = m.addVar(vtype=GRB.BINARY, name="x_rations")

# objective -- total value (pts), written out term by term
m.setObjective(
    9 * x_rifle + 8 * x_boots + 9 * x_helmet + 8 * x_canteen + 3 * x_medkit + 9 * x_rations,
    GRB.MAXIMIZE,
)

# constraint -- total weight (kg) within the 21 kg capacity, written out term by term
m.addConstr(
    5 * x_rifle + 7 * x_boots + 8 * x_helmet + 8 * x_canteen + 8 * x_medkit + 2 * x_rations <= 21,
    name="capacity",
)

m.optimize()

print(f"\nrifle={x_rifle.X:.0f}  boots={x_boots.X:.0f}  helmet={x_helmet.X:.0f}  "
      f"canteen={x_canteen.X:.0f}  medkit={x_medkit.X:.0f}  rations={x_rations.X:.0f}")
print(f"value = {m.ObjVal:.0f} pts")

## Fancier version (same model) — dictionary + `gp.quicksum`

The version above doesn't scale: 20 items means 20 lines of variables and two 20-term sums typed by hand. This equivalent version loops over a dictionary instead, so it works for any number of items without rewriting the model. Left commented out for now — same answer, less typing, less obvious which term is which.

In [ ]:
# items = {
#     "rifle":   (5, 9),   # (weight kg, value pts)
#     "boots":   (7, 8),
#     "helmet":  (8, 9),
#     "canteen": (8, 8),
#     "medkit":  (8, 3),
#     "rations": (2, 9),
# }
# CAPACITY = 21
#
# m2 = gp.Model("knapsack_dict", env=env)
# x = m2.addVars(items.keys(), vtype=GRB.BINARY, name="x")
# m2.setObjective(gp.quicksum(items[j][1] * x[j] for j in items), GRB.MAXIMIZE)
# m2.addConstr(gp.quicksum(items[j][0] * x[j] for j in items) <= CAPACITY, name="capacity")
# m2.optimize()
# print([j for j in items if x[j].X > 0.5], m2.ObjVal)